# Epidemiological Model Assignment - Parameter exploration
**Understanding disease dynamics**

## Setup & Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.integrate import odeint
import seaborn as sns

---
## Part 1: Parameter Analysis Function (50 points)
Using the SIRD model from your practical as a starting point, you will investigate how the recovery rate affects epidemic outcome 
### 1.1 Function implementation
Task: Create a function called analyze_recovery_rates() that systematically explores different recovery rates.

### 1.2 Requirements
- Test recovery rates: gamma_values = [0.05, 0.1, 0.15, 0.2, 0.25]
- For each γ value, calculate:
    - Peak number of infectious individuals
    - Day when peak occurs
    - Total deaths at end of simulation
    - Basic reproduction number (R₀ = β/γ)
- Return results as a formatted pandas DataFrame
- Generate a publication-quality plot showing all epidemic curves

### 1.3 Expected output format
Your function should produce:

- A DataFrame with columns: ['gamma', 'R0', 'peak_infected', 'peak_day', 'total_deaths']
- A matplotlib figure with properly labeled axes, legend, and title

In [ ]:
def analyze_recovery_rates(beta, mu, N, I0, simulation_days, gamma_values):
    """
    Analyze epidemic outcomes for different recovery rates.
    Parameters:
    -----------
    beta : float
        Transmission rate
    mu : float  
        Mortality rate
    N : int
        Total population
    I0 : int
        Initial infected individuals
    simulation_days : int
        Simulation duration in days
    Returns:
    --------
    pandas.DataFrame
        Results summary for each recovery rate
    """
    
    #Initialize lists for future dataframe
    R0_values = []
    peak_infected_values = []
    peak_day_values = []
    total_deaths_values = [] 
    for gamma in gamma_values:
        R0 = beta / gamma  # Calculate Ro for current gamma
        #Initialize populations and deaths
        S = N - I0
        I = I0
        R = 0
        D = 0
        #initialize peak infected variables
        peak_infected = I
        peak_day = 0
        for day in range(simulation_days):
            dSdt = -beta * S * I / N  # Same as SIR
            dIdt = beta * S * I / N - (gamma + mu) * I  # SIR equation minus deaths (- mu * I)
            dRdt = gamma * I  # Same as SIR
            dDdt = mu * I  # Equation for deaths
            # iterating Euler steps
            S = max(0, S + dSdt)
            I = max(0, I + dIdt)
            R = max(0, R + dRdt)
            D = max(0, D + dDdt)
            if I > peak_infected:
                peak_infected = I
                peak_day = day
        #Update lists
        R0_values.append(R0)
        peak_infected_values.append(round(peak_infected))
        peak_day_values.append(peak_day)
        total_deaths_values.append(round(D))
    #format results into a dataframe
    df = pd.DataFrame({
        'gamma': gamma_values,
        'R0': R0_values,
        'peak_infected': peak_infected_values,
        'peak_day': peak_day_values,
        'total_deaths': total_deaths_values
    })
    return df

---
## Part 2: Scenario comparison (30 points)
### 2.1 Scenario analysis
Use your function to compare two scenarios:

**Scenario A - "High Transmission":**

In [1]:
#beta=0.4, mu=0.02, N=1000, I0=5, simulation_days=200 DON'T TOUCH

**Scenario B - "Low Transmission":**

In [2]:
#beta=0.2, mu=0.005, N=1000, I0=5, simulation_days=200 DON'T TOUCH

### 2.2 Deliverables
Create notebook cells that:

1. Run both scenarios using your function
2. Display both result DataFrames
3. Create a comparative visualization (side-by-side plots or combined plot)
4. Write a markdown cell analyzing which scenario is worse for public health and why

In [ ]:
# Table results for the two data frames
print("Results for original gamma values [0.05, 0.1, 0.15, 0.2, 0.25]\n")
data_frame_1 = analyze_recovery_rates(beta=0.4, mu= 0.02, N=1000, I0=5, simulation_days=200, gamma_values=[0.05, 0.1, 0.15, 0.2, 0.25])
data_frame_2 = analyze_recovery_rates(beta=0.2, mu=0.005, N=1000, I0=5, simulation_days=200, gamma_values=[0.05, 0.1, 0.15, 0.2, 0.25])
print(f"{data_frame_1}\n")
print(f"{data_frame_2}\n")
print("Results for modified gamma values [0.075, 0.15, 0.225, 0.3, 0.375]\n")
data_frame_1 = analyze_recovery_rates(beta=0.4, mu= 0.02, N=1000, I0=5, simulation_days=200, gamma_values=[0.075, 0.15, 0.225, 0.3, 0.375])
data_frame_2 = analyze_recovery_rates(beta=0.2, mu=0.005, N=1000, I0=5, simulation_days=200, gamma_values=[0.075, 0.15, 0.225, 0.3, 0.375])
print(f"{data_frame_1}\n")
print(f"{data_frame_2}\n")

Analysis:
The table results show a decreasing trend overall in both peak infected people and total number of deaths as the recovery rate increases for both Scenario A and Scenario B. Scenario B sees R0 < 1 when gamma = 0.25 or less, whereas Scenario A reaches its sharpest decrease in number of peak infected people and total death count when gamma = 0.375.

---
## Part 3: Policy recommendations (20 points)
Create markdown cells with analysis addressing:
### 3.1 Parameter impact analysis
- How does increasing recovery rate affect peak infections, total deaths, and epidemic duration?
- Use specific numbers from your results to support your conclusions

Answer: Increasing recovery rate correlates to decreasing R0, which also decreases peak infected people and total deaths. This appears to be consistent for both Scenario A and Scenario B. Scenario A starts with 545 as its peak infected and 285 total deaths for gamma = 0.05, decreasing up to 65 peak infected and 43 total deaths for gamma = 0.25. As for Scenario B, the peak infected number starts at 381 and total deaths at 88, decreasing to 5 as peak infected and 0 total deaths.

### 3.2 Intervention analysis
- If an intervention could increase recovery rate by 50%, what would be the expected impact on total deaths?
- Use Scenario A as your baseline and show calculations

Answer: Due to the intervention, the recovery rate gets updated (1.5γ). This goes on to affect the basic reproduction number (β/1.5γ = 0.67R₀). We plug into the analysis the new gamma values [0.075, 0.15, 0.225, 0.3, 0.375] and proceed to calculate for both scenarios. With the original gamma values, we can see that Scenario A demonstrates a higher trend overall in total deaths compared to Scenario B. This would be the case due to lower R0, leading to a decrease in peak infected people and therefore, less total deaths. For the modified gamma values, Scenario A sees a slight decreasing trend compared to the original. Scenario B sees a dramatic drop, reaching 0 total deaths as gamma reaches 0.3 and showing R0 < 1 when gamma reaches 0.225. The impact of this intervention reveals how crucial it is for decreasing the death count.
### 3.3 Real-world application
- Name one real medical intervention that could increase recovery rates
-- Explain the mechanism and estimate realistic effectiveness
Answer: 


---
## Technical requirements
### Documentation requirements
- Markdown cells: Clear explanations before each code section
- Code comments: Explain complex calculations and logic
- Plot formatting: All plots must have titles, axis labels, legends, and proper styling
### Results presentation
- Clean dataframes: Well-formatted tables with appropriate column names
- Clear visualizations: Readable plots with consistent styling
- Integrated analysis: Code, results, and interpretation in logical flow
### Tips
1. Start with working code: Ensure your SIRD model function works before building the analysis function
2. Test incrementally: Test your function with one gamma value before running all five
3. Use markdown effectively: Explain your approach and findings clearly between code cells
4. Professional plots: Use consistent colors, proper labels, and clean styling